In [17]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

%matplotlib tk


# --- 1. SYSTEM CONSTANTS ---
mu = 398600.0        # Earth gravitational parameter (km^3/s^2)
Re = 6371.01         # Earth radius (km)
M_E = 5.972e24       # Earth Mass (kg)
ts = 1.0             # Time step (seconds)

# --- 2. MOON EPHEMERIS (Geometric Ellipse Model) ---
P_o, A_o = 60 * Re, 60 * Re
a = (P_o + A_o + 2*Re) / 2
c = a - P_o - Re
e0 = (Re + A_o - Re - P_o) / (2*Re + A_o + P_o)
T_o = 2 * np.pi * np.sqrt(a**3 / mu)
w = 2 * np.pi / T_o

t_moon = np.arange(0, 0.5 * T_o, ts)
Ex1, Ey1, Ez1 = [], [], []

for t in t_moon:
    r0 = (P_o + Re) / (1 + e0 * np.cos(w * (t - 1)))
    d = A_o - P_o
    n1 = (a + c - d) / ((P_o + Re) / (1 + e0 * np.cos(0)))
    Ex1.append(n1 * r0 * np.cos(w * t) + d)
    Ey1.append(n1 * r0 * np.sin(w * t))
    Ez1.append(0)

Ex1, Ey1, Ez1 = np.array(Ex1), np.array(Ey1), np.array(Ez1)

# --- 3. CRAFT INITIAL CONDITIONS (300km Circular Orbit) ---
M_o = 7280.0         # Initial Mass (kg)
M_dot = 0.0          # Mass flow rate
Thrust = 0.0         # Thrust (kN)

bR0 = Re + 300.0     # 300km Altitude
LAT, LON = 0.0, 0.0
pos = np.array([bR0 * np.sin(LON) * np.cos(LAT), 
                bR0 * np.sin(LON) * np.sin(LAT), 
                bR0 * np.cos(LON)])

# Velocity for circular orbit (approx 7.73 km/s)
AV_lat, AV_lon = 0.0, 90 * np.pi / 180
bRv0 = 7.72988176989
vel = np.array([bRv0 * np.sin(AV_lon) * np.cos(AV_lat), 
                bRv0 * np.sin(AV_lon) * np.sin(AV_lat), 
                bRv0 * np.cos(AV_lon)])

# --- 4. ACCELERATION FUNCTION ---
def calc_acceleration(position, moon_idx, mass):
    R_mag = np.linalg.norm(position)
    a_earth = -(mu / R_mag**3) * position
    
    # Lunar Perturbation
    moon_pos = np.array([Ex1[moon_idx], Ey1[moon_idx], Ez1[moon_idx]])
    vec_to_moon = moon_pos - position
    dist_to_moon = np.linalg.norm(vec_to_moon)
    a_moon = ((0.0123031469 * mu) / dist_to_moon**3) * vec_to_moon
    
    # Thrust (Set to 0 for this test, but logic remains)
    a_thrust = np.array([0.0, 0.0, 0.0]) 
    
    return a_earth + a_moon + a_thrust

# --- 5. THE VELOCITY VERLET ENGINE ---
iterations = 3500
time_steps = np.arange(iterations)
alt_history = np.zeros(iterations)
rng_history = np.zeros(iterations)

# Tracking arrays for 3D plot
x_hist, y_hist, z_hist = np.zeros(iterations), np.zeros(iterations), np.zeros(iterations)

current_mass = M_o
acc_old = calc_acceleration(pos, 0, current_mass)
rng_total = 0.0

print("Initiating Symplectic Integration Sequence...")

for i in range(iterations):
    # Store history
    alt_history[i] = np.linalg.norm(pos) - Re
    rng_history[i] = rng_total
    x_hist[i], y_hist[i], z_hist[i] = pos[0], pos[1], pos[2]
    
    # 1. Kinematic Position Update
    pos_new = pos + vel * ts + 0.5 * acc_old * (ts**2)
    
    # Accumulate Range (Distance traveled)
    rng_total += np.linalg.norm(pos_new - pos)
    
    # 2. Calculate New Acceleration
    moon_idx = min(i + 1, len(Ex1) - 1)
    acc_new = calc_acceleration(pos_new, moon_idx, current_mass)
    
    # 3. Kinematic Velocity Update
    vel_new = vel + 0.5 * (acc_old + acc_new) * ts
    
    # Step forward
    pos = pos_new
    vel = vel_new
    acc_old = acc_new

print(f"Simulation Complete. Final Range: {rng_total:.2f} km")

# --- 6. PAPER-READY TELEMETRY PLOTS ---
fig = plt.figure(figsize=(14, 6))
plt.style.use('dark_background') # Keeping your terminal aesthetic

# Plot 1: Altitude over Time
ax1 = fig.add_subplot(121)
ax1.plot(time_steps, alt_history, color='cyan', linewidth=2)
ax1.set_title("Verlet Altitude Profile (3B Model)", fontsize=14)
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Altitude (km)")
ax1.grid(True, alpha=0.3)

# Plot 2: 3D Orbital Path
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(x_hist, y_hist, z_hist, color='yellow', label="Craft Trajectory")
# Simple Earth representation
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
ex = Re * np.cos(u) * np.sin(v)
ey = Re * np.sin(u) * np.sin(v)
ez = Re * np.cos(v)
ax2.plot_wireframe(ex, ey, ez, color='blue', alpha=0.3)
ax2.set_title("Geocentric Trajectory", fontsize=14)
ax2.legend()

plt.tight_layout()
plt.show()

Initiating Symplectic Integration Sequence...
Simulation Complete. Final Range: 27054.38 km


In [16]:
# --- 7. THE BEZIER DETRENDING FILTER ---
print("Applying Bezier Signal Detrending...")

# 1. Generate the normalized time parameter 't' (from 0 to 1)
t_norm = np.linspace(0, 1, iterations)

# 2. Extract the 4 Control Points based on your GlowScript logic
# We use the altitude history (alt_history) to find the drift profile
span = iterations
by1 = alt_history[0]
by2 = alt_history[int(span / 2)]
by3 = alt_history[int(3 * span / 4)]
by4 = alt_history[-1]

# 3. Compute the Cubic Bezier Curve (Vectorized)
# B(t) = (1-t)^3*P0 + 3(1-t)^2*t*P1 + 3(1-t)*t^2*P2 + t^3*P3
bezier_curve = ((1 - t_norm)**3) * by1 + \
               3 * ((1 - t_norm)**2) * t_norm * by2 + \
               3 * (1 - t_norm) * (t_norm**2) * by3 + \
               (t_norm**3) * by4

# 4. Detrend the Signal (Subtracting the Bezier drift from the true altitude)
# We add back the initial altitude (by1) to keep it centered at 300km
detrended_alt = alt_history - bezier_curve + by1

# --- 8. FILTER ANALYSIS PLOT ---
fig2 = plt.figure(figsize=(14, 6))
plt.style.use('dark_background')

ax3 = fig2.add_subplot(111)

# Plotting the raw Verlet data (The expanding perturbation)
ax3.plot(time_steps, alt_history, color='red', alpha=0.5, label='Raw 3B Verlet Alt')

# Plotting the Bezier trendline
ax3.plot(time_steps, bezier_curve, color='purple', linewidth=2, linestyle='--', label='Bezier Drift Filter')

# Plotting the corrected, stationary orbit
ax3.plot(time_steps, detrended_alt, color='cyan', linewidth=2, label='Detrended (Stationary) Alt')

ax3.set_title("3B Perturbation vs. Bezier Stabilized Orbit", fontsize=16)
ax3.set_xlabel("Time (seconds)")
ax3.set_ylabel("Altitude (km)")
ax3.legend(loc="upper left")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Applying Bezier Signal Detrending...
